# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant schema defines each `recordSet` and its corresponding `@id`. Below, we list each available record set along with its fields and columns, referencing all entities solely via their `@id`.

In [ ]:
# List all available record sets and their fields by @id
record_sets = []

if hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet

print("Record Sets and their fields:")
for record_set in record_sets:
    print(f"RecordSet @id: {record_set['@id']}")
    fields = record_set.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        print(f"  Field @id: {field['@id']} | Columns: {[col['@id'] for col in field.get('column', [])] if 'column' in field else []}")

### Example: Preview records from each record set
Below, we show how to preview the actual records using `mlcroissant.Dataset.records()` for a specific record set, referenced only via its `@id`.

In [ ]:
# Preview records for each record set by @id
for record_set in record_sets:
    print(f"Records from RecordSet @id: {record_set['@id']}")
    try:
        for i, x in enumerate(dataset.records(record_set=record_set['@id'])):
            print(x)
            if i >= 2:  # print only first 3 records
                break
    except Exception as e:
        print(f"  Failed to load records: {e}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use exclusively `@id` for referencing record sets and fields.

DataFrames are created for each record set for further processing.

In [ ]:
# Extract data from each record set
dataframes = {}

for record_set in record_sets:
    rs_id = record_set['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Data columns for RecordSet @id={rs_id}:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Failed loading RecordSet @id={rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on specific criteria
- Normalizing numeric fields
- Categorizing data
- Removing outliers
- Grouping by attributes

All processing is done by referencing column `@id`s from the DataFrame.

In [ ]:
# Choose the main record set and numeric fields for analysis
# You may need to select the most populated record set or key fields based on overview above.

# Example: Select first record set
main_rs_id = list(dataframes.keys())[0] if dataframes else None

if main_rs_id:
    df = dataframes[main_rs_id]

    # Print all columns (by @id only)
    print(f"Columns in main RecordSet @id '{main_rs_id}':")
    print(df.columns.tolist())

    # Example: Choose numeric fields by inspecting (e.g., GAD-7 score, PHQ-9 score, PSQ score)
    numeric_fields = [c for c in df.columns if any(s in c.lower() for s in ["gad", "phq", "psq", "score", "age"])]

    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10

        # Filter records
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        if pd.api.types.is_numeric_dtype(df[numeric_field_id]):
            filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping: Use presence of a demographic/group variable (e.g., education, gender, etc. referenced by @id)
        possible_groups = [c for c in df.columns if any(s in c.lower() for s in ["education", "gender", "village", "marital", "level_of_education"])]
        group_field_id = possible_groups[0] if possible_groups else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped averages for {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, using only DataFrame columns referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt

# Visualize a histogram for the main numeric field
if main_rs_id and numeric_fields:
    field = numeric_fields[0]
    plt.figure(figsize=(8,5))
    df[field].dropna().hist(bins=20)
    plt.title(f'Distribution of {field}')
    plt.xlabel(field)
    plt.ylabel('Frequency')
    plt.show()

# Example: Bar plot of group averages if grouped_df exists
if 'grouped_df' in locals():
    plt.figure(figsize=(10,7))
    plt.bar(grouped_df[group_field_id].astype(str), grouped_df[numeric_field_id])
    plt.title(f'{numeric_field_id} Mean by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset loaded successfully using `mlcroissant`.
- All data entities were referenced solely by their `@id` as defined in the Croissant schema, ensuring semantic traceability.
- Key demographic and mental health fields can be filtered and analyzed.
- Data normalization and grouping allow for meaningful aggregation (e.g., average scores by educational level or gender).
- Visualizations reveal distribution patterns and group differences in the survey data.

For further studies, consider advanced modeling or linkage with community interventions.